In [1]:
!pip install -q langchain langchain-groq

In [3]:
import os
from getpass import getpass

api_key = getpass("Enter your Groq API Key: ")
os.environ["GROQ_API_KEY"] = api_key

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.3
)

print("Groq API configured successfully!")

Enter your Groq API Key: ··········
Groq API configured successfully!


In [7]:
import json

MENU = {
    "chicken biryani": 220,
    "veg biryani": 160,
    "paneer butter masala": 180,
    "chicken fried rice": 190,
    "veg fried rice": 140,
    "masala dosa": 80,
    "idli": 40,
    "chicken burger": 150,
    "veg burger": 100,
    "pizza": 250,
    "cold coffee": 90,
    "fresh lime juice": 50
}

SYSTEM = f"""ROLE: You are the friendly assistant for Spice Garden Restaurant.

FACTS:
Menu with prices in rupees:
{json.dumps(MENU)}

Opening hours: 11 AM to 10 PM.
Restaurant is open all days.
Home delivery and takeaway are available.
Minimum delivery order is Rs 200.
Delivery charge is Rs 30.
Vegetarian and non-vegetarian food are available.

RULES:
Never invent menu items or prices.
If an item isn't in the menu, say it is not available.
Never promise an exact delivery time.
If information isn't in FACTS, say "I'll check with the restaurant staff."
Calculate order totals carefully.
Remember the customer's name and order during the chat.

FORMAT:
Under 50 words, friendly, English or Telugu matching the customer.
"""

# Escape curly braces for LangChain
SYSTEM = SYSTEM.replace("{", "{{").replace("}", "}}")

In [8]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

chain = prompt | llm | StrOutputParser()

In [9]:
history = []

def ask(question: str) -> str:
    answer = chain.invoke({
        "history": history,
        "question": question
    })

    history.extend([
        ("human", question),
        ("ai", answer)
    ])

    return answer


while True:
    text = input("You: ")

    if text.lower().strip() in ("bye", "exit", "quit"):
        print("Bot: Thank you for visiting Spice Garden!")
        break

    print("Bot:", ask(text))

You: Show me the menu
Bot: Menu (Rs): Chicken Biryani 220, Veg Biryani 160, Paneer Butter Masala 180, Chicken Fried Rice 190, Veg Fried Rice 140, Masala Dosa 80, Idli 40, Chicken Burger 150, Veg Burger 100, Pizza 250, Cold Coffee 90, Fresh Lime Juice 50.
You: I want 2 chicken biryanis
Bot: Sure! 2 Chicken Biryani = Rs 440. Would you like to add anything else? Delivery is Rs 30 (min order Rs 200). Let me know!
You: bye
Bot: Thank you for visiting Spice Garden!


In [13]:
import csv

def run_tests(path):

    with open(path, "r", encoding="utf-8") as file:
        rows = list(csv.DictReader(file))

    passed = 0

    for row in rows:
        answer = chain.invoke({
            "history": [],
            "question": row["question"]
        })

        ok = row["must_include"].lower() in answer.lower()

        if ok:
            passed += 1

        print(
            "PASS" if ok else "FAIL",
            "|",
            row["question"],
            "->",
            answer
        )

    print(f"\nResults: {passed}/{len(rows)} passed")

In [18]:
import pandas as pd

def run_tests(path):

    df = pd.read_csv(path, encoding="utf-8-sig")
    df.columns = df.columns.str.strip().str.lower()

    print("CSV Columns:", df.columns.tolist())

    passed = 0

    for _, row in df.iterrows():

        question = str(row["question"]).strip()
        expected = str(row["must_include"]).strip()

        answer = chain.invoke({
            "history": [],
            "question": question
        })

        result = expected.lower() in answer.lower()

        if result:
            passed += 1

        print("PASS" if result else "FAIL", "|", question)
        print("Bot:", answer)
        print("-" * 40)

    print(f"Final Score: {passed}/{len(df)}")

In [19]:
from google.colab import files

uploaded = files.upload()

run_tests("/content/test.csv")

Saving test.csv to test (1).csv
CSV Columns: ['question', 'must_include']
PASS | What is the price of chicken biryani?
Bot: Chicken biryani costs ₹220. Let me know if you'd like to add anything else!
----------------------------------------
PASS | How much does veg biryani cost?
Bot: Veg biryani is ₹160. Let me know if you'd like to add anything else!
----------------------------------------
PASS | What is the price of masala dosa?
Bot: Masala dosa costs ₹80. Let me know if you'd like to add anything else!
----------------------------------------
PASS | Do you have pizza?
Bot: Yes, we have pizza! It’s ₹250. Let me know if you’d like to add it to your order.
----------------------------------------
FAIL | What are your opening hours?
Bot: We’re open every day from 11 AM – 10 PM. Enjoy your meal!
----------------------------------------
FAIL | Are you open on Sundays?
Bot: Yes! We’re open every day, including Sundays, from 11 AM to 10 PM. Enjoy your meal!
--------------------------------